# TB macro model on summer4

Three age bands, ragged clinical and infectious strata on `active` only,
run from 1800 to 1999. `passive_detection` and `detection` both move people
into treatment; both are kept so the run matches the summer3wip golden.

The calibration cell is the user gate: the posterior for `contact_rate`
should cover the synthetic truth 0.25.

In [ ]:
import numpy as np
import plotly.graph_objects as go

from tb_macro.calibration import sample, synthetic_target
from tb_macro.constants import BASE_PARAMS, CALIBRATION_CONTACT_RATE, START_YEAR
from tb_macro.model import run
from tb_macro.outputs import disease_state_totals, infection_target

result = run(BASE_PARAMS)
totals = {name: np.asarray(values) for name, values in disease_state_totals(result).items()}
years = np.arange(START_YEAR, START_YEAR + len(next(iter(totals.values()))))
pop = np.stack(list(totals.values()), axis=0).sum(axis=0)
assert np.allclose(pop, 3000.0, rtol=1e-4, atol=1.0)
target = np.asarray(infection_target(result))
assert target.shape == (8,)
assert np.all(np.isfinite(target))
print("population conserved, target length", target.shape[0])

In [ ]:
fig = go.Figure()
for name, values in totals.items():
    fig.add_trace(go.Scatter(x=years, y=values, name=name))
fig.update_layout(
    title="Compartments by disease state",
    xaxis_title="year",
    yaxis_title="people",
)
fig.show()

## Calibration

Synthetic target: the rolling infection series at `contact_rate = 0.25`,
fuzzed with a seeded `exp(N(0, 0.01))`. Prior `Uniform(0.001, 0.5)`,
Poisson factor as in the original notebook, NUTS 200/200 × 4 chains.
Chains are sequential because `CompiledModel.run` is not vmap-safe.

In [ ]:
import arviz as az

observed = synthetic_target(seed=0)
assert observed.shape == target.shape
mcmc = sample(observed, num_warmup=200, num_samples=200, num_chains=4, seed=0)
idata = az.from_numpyro(mcmc)
print(az.summary(idata, var_names=["contact_rate"]))
draws = np.asarray(mcmc.get_samples()["contact_rate"])
lo, hi = np.quantile(draws, [0.025, 0.975])
assert lo <= CALIBRATION_CONTACT_RATE <= hi
az.plot_posterior(idata, var_names=["contact_rate"])
az.plot_trace(idata, var_names=["contact_rate"], compact=False)